In [7]:
import gmsh

gmsh.initialize()
gmsh.model.add("wire_force_contour")

# --------------------------------------------------
# Parameters
# --------------------------------------------------

L = 2.0

r = 0.05
r_force = 0.10

cx = L / 2
cy = L / 2

distance = 0.40
x1 = cx - distance / 2

lc_air = 0.05
lc_wire = 0.01

# --------------------------------------------------
# Geometry
# --------------------------------------------------

square = gmsh.model.occ.addRectangle(0, 0, 0, L, L)

wire1 = gmsh.model.occ.addDisk(x1, cy, 0, r, r)

force1 = gmsh.model.occ.addDisk(x1, cy, 0, r_force, r_force)

# --------------------------------------------------
# Air = square - force disk
# --------------------------------------------------

air, _ = gmsh.model.occ.cut(
    [(2, square)],
    [(2, force1)],
    removeObject=True,
    removeTool=False,
)

# --------------------------------------------------
# Shell = force disk - wire
# --------------------------------------------------

shell1, _ = gmsh.model.occ.cut(
    [(2, force1)],
    [(2, wire1)],
    removeObject=True,
    removeTool=False,
)

gmsh.model.occ.synchronize()

# --------------------------------------------------
# Physical surfaces
# --------------------------------------------------

gmsh.model.addPhysicalGroup(2, [air[0][1]], name="air")
gmsh.model.addPhysicalGroup(2, [shell1[0][1]], name="shell1")
gmsh.model.addPhysicalGroup(2, [wire1], name="wire1")

# --------------------------------------------------
# Boundary curves
# --------------------------------------------------

air_curves = gmsh.model.getBoundary(
    [(2, air[0][1])],
    oriented=False
)

shell_curves = gmsh.model.getBoundary(
    [(2, shell1[0][1])],
    oriented=False
)

# --------------------------------------------------
# Classify curves by radius
# --------------------------------------------------

outer_boundary = []
force_curve = None

for dim, tag in air_curves:

    xmin, ymin, _, xmax, ymax, _ = gmsh.model.getBoundingBox(dim, tag)

    dx = xmax - xmin
    dy = ymax - ymin

    # square edges
    if dx > 0.5 * L or dy > 0.5 * L:
        outer_boundary.append(tag)
    else:
        force_curve = tag

gmsh.model.addPhysicalGroup(
    1,
    outer_boundary,
    name="boundary"
)

gmsh.model.addPhysicalGroup(
    1,
    [force_curve],
    name="force1"
)

# --------------------------------------------------
# Mesh size
# --------------------------------------------------

gmsh.model.mesh.setSize(
    gmsh.model.getEntities(0),
    lc_air,
)

wire_pts = gmsh.model.getBoundary(
    [(2, wire1)],
    recursive=True,
)

gmsh.model.mesh.setSize(
    wire_pts,
    lc_wire,
)

# --------------------------------------------------
# Mesh
# --------------------------------------------------

gmsh.model.mesh.generate(2)

gmsh.write("wire_force_contour.msh")

gmsh.fltk.run()

gmsh.finalize()